In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

token = os.environ.get("OPENAI_API_KEY")

if token:
    masked = f"{token[:4]}...{token[-4:]}" if len(token) > 8 else "****"
    print(f"OPENAI_API_KEY loaded ({len(token)} characters): {masked}")
else:
    print("OPENAI_API_KEY not found. Check that .env exists in this directory, "
          "the key is spelled exactly 'OPENAI_API_KEY', and load_dotenv() ran without error.")

OPENAI_API_KEY loaded (164 characters): sk-p...VmEA


In [ ]:
"""
Few-shot relevance classification using the standard OpenAI API.

Same crash-safe / resumable design as classify_with_open_llm.py: writes each
result immediately, skips already-processed wos_ids on rerun.

Requires: pip install openai
Requires environment variable: OPENAI_API_KEY
"""

import json
import os
import re
import time
import pandas as pd
from openai import OpenAI

# ---- Config ----
MODEL_NAME = "gpt-4.1"  # or "gpt-5.6-terra"gpt-4o", "gpt-5", etc. -- set to whichever
                          # model you have access to / want to compare
PROMPT_FILE = "gpt_classification_prompt.txt"
INPUT_FILE = "merged_shuffled.xlsx"        # columns: wos_id, title, abstract
OUTPUT_JSONL = "classification_results_gpt4.jsonl"
MAX_RETRIES = 3
RETRY_DELAY_SECONDS = 5


def get_client() -> OpenAI:
    return OpenAI(api_key=os.environ["OPENAI_API_KEY"])


def load_system_prompt(path: str) -> str:
    with open(path) as f:
        return f.read()


def load_input_papers(path: str) -> pd.DataFrame:
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    try:
        return pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp1252")


def already_processed_ids(output_path: str) -> set:
    if not os.path.exists(output_path):
        return set()
    ids = set()
    with open(output_path) as f:
        for line in f:
            try:
                ids.add(json.loads(line)["wos_id"])
            except (json.JSONDecodeError, KeyError):
                continue
    return ids


def extract_json(text: str) -> dict | None:
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def build_user_message(title: str, abstract: str) -> str:
    return f"Title: {title}\nAbstract: {abstract}"


def call_with_retries(client, system_prompt, user_message):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_message},
                ],
                temperature=0,
                max_tokens=300,
            )
            return response.choices[0].message.content
        except Exception as e:
            last_error = e
            print(f"  Attempt {attempt} failed: {e}. Retrying in {RETRY_DELAY_SECONDS}s...")
            time.sleep(RETRY_DELAY_SECONDS)
    raise last_error


def main():
    client = get_client()
    system_prompt = load_system_prompt(PROMPT_FILE)
    papers = load_input_papers(INPUT_FILE)
    done_ids = already_processed_ids(OUTPUT_JSONL)
    print(f"Total papers: {len(papers)} | Already processed: {len(done_ids)}")

    n_ok, n_parse_failed, n_api_failed = 0, 0, 0
    flagged = []

    with open(OUTPUT_JSONL, "a") as out_f:
        for row in papers.itertuples():
            wos_id = row.wos_id
            if wos_id in done_ids:
                continue

            user_message = build_user_message(row.title, row.abstract)
            try:
                generated = call_with_retries(client, system_prompt, user_message)
            except Exception as e:
                print(f"  FAILED after retries for {wos_id}: {e}")
                n_api_failed += 1
                flagged.append(wos_id)
                continue

            parsed = extract_json(generated)
            result = {"wos_id": wos_id, "title": row.title, "raw_model_output": generated}
            if parsed is not None and "label" in parsed:
                result.update(parsed)
                n_ok += 1
            else:
                result.update({"label": None, "trap_reason": None, "reason": None, "parse_failed": True})
                n_parse_failed += 1
                flagged.append(wos_id)

            out_f.write(json.dumps(result) + "\n")
            out_f.flush()

    print(f"\nDone. OK: {n_ok} | Parse failed: {n_parse_failed} | API failed: {n_api_failed}")
    if flagged:
        print(f"Flagged wos_ids: {flagged[:20]}{' ...' if len(flagged) > 20 else ''}")
    print(f"Results saved to {OUTPUT_JSONL}")


if __name__ == "__main__":
    main()

Total papers: 433 | Already processed: 0
  Attempt 1 failed: Error code: 400 - {'error': {'message': "Unsupported parameter: 'max_tokens' is not supported with this model. Use 'max_completion_tokens' instead.", 'type': 'invalid_request_error', 'param': 'max_tokens', 'code': 'unsupported_parameter'}}. Retrying in 5s...
  Attempt 2 failed: Error code: 400 - {'error': {'message': "Unsupported parameter: 'max_tokens' is not supported with this model. Use 'max_completion_tokens' instead.", 'type': 'invalid_request_error', 'param': 'max_tokens', 'code': 'unsupported_parameter'}}. Retrying in 5s...


# TERRA

In [1]:
"""
Few-shot relevance classification using the standard OpenAI API.

Same crash-safe / resumable design as classify_with_open_llm.py: writes each
result immediately, skips already-processed wos_ids on rerun.

Requires: pip install openai
Requires environment variable: OPENAI_API_KEY
"""

import json
import os
import re
import time
import pandas as pd
from openai import OpenAI

# ---- Config ----
MODEL_NAME = "gpt-5.6-luna" # same model used for KG triple extraction --
    # note: this is a reasoning-capable model (supports reasoning.effort),
    # so the max_tokens headroom below matters, same consideration as Sonnet
                          # model you have access to / want to compare
PROMPT_FILE = "gpt_classification_prompt.txt"
INPUT_FILE = "merged_shuffled.xlsx"        # columns: wos_id, title, abstract
OUTPUT_JSONL = "classification_results_terra.jsonl"
MAX_RETRIES = 3
RETRY_DELAY_SECONDS = 5


def get_client() -> OpenAI:
    return OpenAI(api_key=os.environ["OPENAI_API_KEY"])


def load_system_prompt(path: str) -> str:
    with open(path) as f:
        return f.read()


def load_input_papers(path: str) -> pd.DataFrame:
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    try:
        return pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp1252")


def already_processed_ids(output_path: str) -> set:
    if not os.path.exists(output_path):
        return set()
    ids = set()
    with open(output_path) as f:
        for line in f:
            try:
                ids.add(json.loads(line)["wos_id"])
            except (json.JSONDecodeError, KeyError):
                continue
    return ids


def extract_json(text: str) -> dict | None:
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def build_user_message(title: str, abstract: str) -> str:
    return f"Title: {title}\nAbstract: {abstract}"


def call_with_retries(client, system_prompt, user_message):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_message},
                ],
                temperature=1,
                max_completion_tokens=600,  # raised from 300 as cheap insurance -- if
                                   # MODEL_NAME is a reasoning-tier model
                                   # (gpt-5, o1, o3), hidden reasoning tokens
                                   # can consume the budget before the visible
                                   # JSON answer is produced, same issue
                                   # already hit and fixed for Sonnet
            )
            return response.choices[0].message.content
        except Exception as e:
            last_error = e
            print(f"  Attempt {attempt} failed: {e}. Retrying in {RETRY_DELAY_SECONDS}s...")
            time.sleep(RETRY_DELAY_SECONDS)
    raise last_error


def main():
    client = get_client()
    system_prompt = load_system_prompt(PROMPT_FILE)
    papers = load_input_papers(INPUT_FILE)
    done_ids = already_processed_ids(OUTPUT_JSONL)
    print(f"Total papers: {len(papers)} | Already processed: {len(done_ids)}")

    n_ok, n_parse_failed, n_api_failed = 0, 0, 0
    flagged = []

    with open(OUTPUT_JSONL, "a") as out_f:
        for row in papers.itertuples():
            wos_id = row.wos_id
            if wos_id in done_ids:
                continue

            user_message = build_user_message(row.title, row.abstract)
            try:
                generated = call_with_retries(client, system_prompt, user_message)
            except Exception as e:
                print(f"  FAILED after retries for {wos_id}: {e}")
                n_api_failed += 1
                flagged.append(wos_id)
                continue

            parsed = extract_json(generated)
            result = {"wos_id": wos_id, "title": row.title, "raw_model_output": generated}
            if parsed is not None and "label" in parsed:
                result.update(parsed)
                n_ok += 1
            else:
                result.update({"label": None, "trap_reason": None, "reason": None, "parse_failed": True})
                n_parse_failed += 1
                flagged.append(wos_id)

            out_f.write(json.dumps(result) + "\n")
            out_f.flush()

    print(f"\nDone. OK: {n_ok} | Parse failed: {n_parse_failed} | API failed: {n_api_failed}")
    if flagged:
        print(f"Flagged wos_ids: {flagged[:20]}{' ...' if len(flagged) > 20 else ''}")
    print(f"Results saved to {OUTPUT_JSONL}")


if __name__ == "__main__":
    main()

Total papers: 433 | Already processed: 55

Done. OK: 378 | Parse failed: 0 | API failed: 0
Results saved to classification_results_terra.jsonl


In [3]:
"""
Convert a classification results .jsonl file (from classify_with_open_llm.py,
or your dpo_pairs.jsonl / train_pairs.jsonl / validation_pairs.jsonl) into an
.xlsx file for easy review in Excel.

Usage: edit INPUT_PATH and OUTPUT_PATH below, then run.
"""

import json
import pandas as pd

INPUT_PATH = "classification_results_terra.jsonl"
OUTPUT_PATH = "LLM-FULL/classification_results_terra.xlsx"


def main():
    rows = []
    with open(INPUT_PATH) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))

    df = pd.DataFrame(rows)
    df.to_excel(OUTPUT_PATH, index=False)
    print(f"Converted {len(rows)} rows from {INPUT_PATH} -> {OUTPUT_PATH}")
    print(f"Columns: {list(df.columns)}")


if __name__ == "__main__":
    main()

Converted 433 rows from classification_results_terra.jsonl -> LLM-FULL/classification_results_terra.xlsx
Columns: ['wos_id', 'title', 'raw_model_output', 'label', 'trap_reason', 'reason']
